In [ ]:
import json
import numpy as np
import torch

In [ ]:
prova = np.load("subset_test_1000/crops/111_2d3ed1a61471caa941876befb9b70328fb40bf42c66ba80d5d4817b43b556d41_512.npy")
prova

In [ ]:
provadue = torch.load("subset_test_1000/embeddings/111.pt")
provadue.size()

In [ ]:
with open("subset_test_1000/json/111_SEMMetaData.json") as f:
    provatre = json.load(f)

len(provatre)

In [ ]:
provatre

In [ ]:
from matplotlib import pyplot as plt

plt.imshow(prova, cmap='gray')
plt.show()

AP_WD  
AP_ALIAS_WD  
AP_FREE_WD  
AP_BEAM_TIME  
AP_STAGE_LOW_Z  
AP_STAGE_CENTRE_X  
AP_C3  
AP_EHT_CAL_30KV  
AP_EHT_CAL_10KV  
AP_EHT_CAL_100  
AP_EHT_CAL_24KV  
AP_MINMAG  

In [ ]:
import os
import json

folder_path = 'subset_test_1000/json'
all_data = []

# Loop through all files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        
        # Open and load each JSON file
        with open(file_path, 'r') as file:
            data = json.load(file)
            new = {
                'SEMImageID': data['SEMImageID'],
                'ImageOriginalName': data['ImageOriginalName'],
                'ImageHashName': data['ImageHashName'],
                'ImageAVGHashName': data['ImageAVGHashName'],
                'AP_WD': data['AP_WD'],
                'AP_ALIAS_WD': data['AP_ALIAS_WD'],
                'AP_FREE_WD': data['AP_FREE_WD'],
                'AP_BEAM_TIME': data['AP_BEAM_TIME'],
                'AP_STAGE_LOW_Z': data['AP_STAGE_LOW_Z'],
                'AP_STAGE_CENTRE_X': data['AP_STAGE_CENTRE_X'],
                'AP_C3': data['AP_C3'],
                'AP_EHT_CAL_30KV': data['AP_EHT_CAL_30KV'],
                'AP_EHT_CAL_10KV': data['AP_EHT_CAL_10KV'],
                'AP_EHT_CAL_100': data['AP_EHT_CAL_100'],
                'AP_EHT_CAL_24KV': data['AP_EHT_CAL_24KV'],
                'AP_MINMAG': data['AP_MINMAG'],
            }
            all_data.append(new)

print(f"Loaded {len(all_data)} files.")

In [ ]:
all_data[0]

In [ ]:
for dicts in all_data:
    if dicts['SEMImageID'] == 111:
        mydict = dicts
        break

mydict

# Metadata Extraction

In [ ]:
"""
Load SEM metadata JSON files and extract the 12 settings fields as
clean numeric values (+ units), instead of raw 'Label = value unit' strings.
"""
import os
import re
import json
import pandas as pd

FOLDER_PATH = "subset_test_1000/json"

SETTINGS_KEYS = [
    "AP_WD",
    "AP_ALIAS_WD",
    "AP_FREE_WD",
    "AP_BEAM_TIME",
    "AP_STAGE_LOW_Z",
    "AP_STAGE_CENTRE_X",
    "AP_C3",
    "AP_EHT_CAL_30KV",
    "AP_EHT_CAL_10KV",
    "AP_EHT_CAL_100",
    "AP_EHT_CAL_24KV",
    "AP_MINMAG",
]

# Matches: "<label> = <number> <unit>"  e.g. "WD = 3.198 mm"
_NUM_PATTERN = re.compile(r"=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*([^\s=]*)\s*$")


def parse_value(raw_string):
    """
    Turn 'WD = 3.198 mm' into (3.198, 'mm').
    Returns (None, None) if the string can't be parsed (e.g. field is None
    or empty in the source JSON).
    """
    if not raw_string:
        return None, None
    match = _NUM_PATTERN.search(raw_string)
    if not match:
        return None, None
    return float(match.group(1)), match.group(2)


def load_all_metadata(folder_path=FOLDER_PATH):
    all_rows = []

    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r") as f:
            data = json.load(f)

        row = {
            "SEMImageID": data["SEMImageID"],
            "ImageOriginalName": data["ImageOriginalName"],
            "ImageHashName": data["ImageHashName"],
            "ImageAVGHashName": data["ImageAVGHashName"],  # bug fix: was hardcoded before
        }

        for key in SETTINGS_KEYS:
            value, unit = parse_value(data.get(key))
            row[key] = value # numeric column, e.g. AP_WD = 3.198
            row[f"{key}_unit"] = unit # e.g. AP_WD_unit = "mm"

        all_rows.append(row)

    df = pd.DataFrame(all_rows)
    print(f"Loaded {len(df)} files.")
    return df


df = load_all_metadata()
print(df[["SEMImageID"] + SETTINGS_KEYS].head())
print()
print("Null counts per settings field (check for parsing failures or missing data):")
print(df[SETTINGS_KEYS].isnull().sum())

In [ ]:
sorted_df = df.sort_values(by='SEMImageID', ascending=True).reset_index(drop=True)
sorted_df.head()